# indlæsning af data og libs.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [2]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
internet = pd.read_csv("data/share-of-individuals-using-the-internet.csv")
tfp = pd.read_csv("data/total-factor-productivity.csv") 
patens = pd.read_csv("data/patent-applications-per-million.csv")
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")

In [3]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

internet = internet.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Share of the population using the Internet": "internet_use"
})

tfp = tfp.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total factor productivity level": "tfp"
})

patens = patens.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Patent applications per million people": "patens"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

# lav begrænsing og merging

In [4]:
start = 2001
end = 2020

lagged_odr = odr.copy()

for df in [odr, tfp, internet, patens, schooling]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

odr = odr[(odr["year"] >= start) & (odr["year"] <= end)]
tfp = tfp[(tfp["year"] >= start) & (tfp["year"] <= end)]
internet = internet[(internet["year"] >= start) & (internet["year"] <= end)]
patens = patens[(patens["year"] >= start) & (patens["year"] <= end)]
schooling = schooling[(schooling["year"] >= start) & (schooling["year"] <= end)]

In [5]:
panel = internet[["country", "code", "year", "internet_use"]].merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    tfp[["code", "year", "tfp"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    patens[["code", "year", "patens"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

#panel.head(30)

In [6]:
# Vælg de variabler, der skal være til stede
required_vars = ["odr", "tfp", "internet_use", "patens", "schooling"]   # tilføj evt. flere

# Marker rækker hvor alle nødvendige variabler findes
panel["complete_case"] = panel[required_vars].notna().all(axis=1)

# Tæl hvor mange unikke lande der har komplette data i hvert år
countries_per_year = (
    panel.loc[panel["complete_case"]]
    .groupby("year")["code"]
    .nunique()
    .reset_index(name="n_countries_complete")
    .sort_values("year")
)

print(countries_per_year)

    year  n_countries_complete
0   2001                    75
1   2002                    77
2   2003                    75
3   2004                    75
4   2005                    79
5   2006                    75
6   2007                    82
7   2008                    80
8   2009                    78
9   2010                    80
10  2011                    83
11  2012                    82
12  2013                    86
13  2014                    88
14  2015                    88
15  2016                    91
16  2017                    92
17  2018                    93
18  2019                    89
19  2020                    89


In [7]:
expected_years = end - start + 1

# behold kun lande der har alle år
balanced_codes = (
    panel.groupby("code")["year"]
    .nunique()
    .loc[lambda x: x == expected_years]
    .index
)

panel_balanced = panel[panel["code"].isin(balanced_codes)].copy()

# sortér pænt
panel_balanced = panel_balanced.sort_values(["code", "year"]).reset_index(drop=True)

print("Antal lande:", panel_balanced["code"].nunique())
print("Antal år pr. land:")
print(panel_balanced.groupby("code")["year"].nunique().value_counts())


Antal lande: 56
Antal år pr. land:
year
20    56
Name: count, dtype: int64


# make mean-intervals

In [8]:
#make a copy of the dataframe
panel_2yr = panel_balanced.copy()

# Lav 2-års perioder
panel_2yr["period_start"] = start + 2 * ((panel_2yr["year"] - start) // 2)
panel_2yr["period_end"] = panel_2yr["period_start"] + 1

# Variabler der skal gennemsnittes
vars_to_average = [
    "tfp",
    "odr",
    "hc", 
    "internet_use",
    "patens",
    "schooling"
]

# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_2yr.columns]

# Collapse til én række pr. land og 2-års periode
panel_2yr = (
    panel_2yr
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_2yr["year"] = panel_2yr["period_start"]

panel_2yr["log_internet_use"] = np.log(panel_2yr["internet_use"])
panel_2yr["log_tfp"] = np.log(panel_2yr["tfp"])
panel_2yr["log_patens"] = np.log(panel_2yr["patens"])

In [9]:
#make a copy of the dataframe
panel_4yr = panel_balanced.copy()

# Lav 4-års perioder
panel_4yr["period_start"] = start + 4 * ((panel_4yr["year"] - start) // 4)
panel_4yr["period_end"] = panel_4yr["period_start"] + 3
# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_2yr.columns]

# Collapse til én række pr. land og 4-års periode
panel_4yr = (
    panel_4yr
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_4yr["year"] = panel_4yr["period_start"]

panel_4yr["log_internet_use"] = np.log(panel_4yr["internet_use"])
panel_4yr["log_tfp"] = np.log(panel_4yr["tfp"])
panel_4yr["log_patens"] = np.log(panel_4yr["patens"])

In [10]:
#make a copy of the dataframe
panel_5yr = panel_balanced.copy()

# Lav 5-års perioder
panel_5yr["period_start"] = start + 5 * ((panel_5yr["year"] - start) // 5)
panel_5yr["period_end"] = panel_5yr["period_start"] + 4
# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_2yr.columns]

# Collapse til én række pr. land og 5-års periode
panel_5yr = (
    panel_5yr
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_5yr["year"] = panel_5yr["period_start"]

panel_5yr["log_internet_use"] = np.log(panel_5yr["internet_use"])
panel_5yr["log_tfp"] = np.log(panel_5yr["tfp"])
panel_5yr["log_patens"] = np.log(panel_5yr["patens"])

In [11]:
panel_balanced["log_internet_use"] = np.log(panel_balanced["internet_use"])
panel_balanced["log_tfp"] = np.log(panel_balanced["tfp"])
panel_balanced["log_patens"] = np.log(panel_balanced["patens"])

# modeller

##### $log(tfp)_it = ODR_it + internet_usage+ patens_it + u_it$

In [12]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model1 = smf.ols(
    "log_tfp ~ odr + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results1 = model1.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results1 = pd.DataFrame({
    "coef": results1.params[["odr", "log_internet_use", "patens"]],
    "std_err": results1.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results1.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results1.round(4))

                    coef  std_err  p_value
odr               0.0114   0.0045   0.0115
log_internet_use  0.0685   0.0225   0.0024
patens            0.0001   0.0001   0.0292


In [13]:
# kopi af det færdige balanced panel
df_est = panel_2yr.copy()

# estimation
model2 = smf.ols(
    "log_tfp ~ odr + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results2 = model2.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results2 = pd.DataFrame({
    "coef": results2.params[["odr", "log_internet_use", "patens"]],
    "std_err": results2.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results2.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results2.round(4))

                    coef  std_err  p_value
odr               0.0116   0.0048   0.0154
log_internet_use  0.0696   0.0242   0.0040
patens            0.0001   0.0001   0.0335


In [14]:
# kopi af det færdige balanced panel
df_est = panel_4yr.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "log_internet_use", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results3.round(4))

                    coef  std_err  p_value
odr               0.0115   0.0052   0.0285
log_internet_use  0.0695   0.0261   0.0077
patens            0.0001   0.0001   0.0451


In [15]:
# kopi af det færdige balanced panel
df_est = panel_5yr.copy()

# estimation
model4 = smf.ols(
    "log_tfp ~ odr + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results4 = model4.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results4 = pd.DataFrame({
    "coef": results4.params[["odr", "log_internet_use", "patens"]],
    "std_err": results4.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results4.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results4.round(4))

                    coef  std_err  p_value
odr               0.0115   0.0056   0.0395
log_internet_use  0.0700   0.0275   0.0110
patens            0.0001   0.0001   0.0427


#### results

In [16]:
results = summary_col(
    [results1, results2, results3, results4],
    stars=True,
    model_names=['1 year', '2 years', '4 years', '5 years'],
    regressor_order=['odr', 'log_internet_use', 'patens'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

#print the model
print("log(tfp) ~ odr + log(internet_use) + patens + FE(code) + FE(year)")
print(results)

log(tfp) ~ odr + log(internet_use) + patens + FE(code) + FE(year)

                   1 year   2 years   4 years  5 years 
-------------------------------------------------------
odr              0.0114**  0.0116**  0.0115**  0.0115**
                 (0.0045)  (0.0048)  (0.0052)  (0.0056)
log_internet_use 0.0685*** 0.0696*** 0.0695*** 0.0700**
                 (0.0225)  (0.0242)  (0.0261)  (0.0275)
patens           0.0001**  0.0001**  0.0001**  0.0001**
                 (0.0001)  (0.0001)  (0.0001)  (0.0001)
R-squared        0.7395    0.7450    0.7587    0.7659  
R-squared Adj.   0.7202    0.7103    0.6898    0.6778  
N                1120      560       280       224     
R2               0.739     0.745     0.759     0.766   
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


#### $log(tfp)_it = ODR_it + internet_usage+ patens_it + u_it$

In [17]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model1 = smf.ols(
    "log_tfp ~ odr + patens + C(code) + C(year)",
    data=df_est
)

results1 = model1.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results1 = pd.DataFrame({
    "coef": results1.params[["odr", "patens"]],
    "std_err": results1.bse[["odr", "patens"]],
    "p_value": results1.pvalues[["odr", "patens"]],
})

print(main_results1.round(4))

          coef  std_err  p_value
odr     0.0035   0.0041   0.4011
patens  0.0001   0.0001   0.1227


In [18]:
# kopi af det færdige balanced panel
df_est = panel_2yr.copy()

# estimation
model2 = smf.ols(
    "log_tfp ~ odr + patens + C(code) + C(year)",
    data=df_est
)

results2 = model2.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results2  = pd.DataFrame({
    "coef": results2.params[["odr", "patens"]],
    "std_err": results2.bse[["odr", "patens"]],
    "p_value": results2.pvalues[["odr", "patens"]],
})

print(main_results2.round(4))

          coef  std_err  p_value
odr     0.0035   0.0043   0.4208
patens  0.0001   0.0001   0.1327


In [19]:
# kopi af det færdige balanced panel
df_est = panel_4yr.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "patens"]],
    "std_err": results3.bse[["odr", "patens"]],
    "p_value": results3.pvalues[["odr", "patens"]],
})

print(main_results3.round(4))

          coef  std_err  p_value
odr     0.0033   0.0047   0.4753
patens  0.0001   0.0001   0.1525


In [20]:
# kopi af det færdige balanced panel
df_est = panel_5yr.copy()

# estimation
model4 = smf.ols(
    "log_tfp ~ odr + patens + C(code) + C(year)",
    data=df_est
)

results4 = model4.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results4 = pd.DataFrame({
    "coef": results4.params[["odr", "patens"]],
    "std_err": results4.bse[["odr", "patens"]],
    "p_value": results4.pvalues[["odr", "patens"]],
})

print(main_results4.round(4))

          coef  std_err  p_value
odr     0.0033   0.0050   0.5123
patens  0.0001   0.0001   0.1490


### Results

In [21]:
results = summary_col(
    [results1, results2, results3, results4],
    stars=True,
    model_names=['1 year', '2 years', '4 years', '5 years'],
    regressor_order=['odr', 'log_internet_use', 'patens'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

#print the model
print("log(tfp) ~ odr + patens + FE(code) + FE(year)")
print(results)

log(tfp) ~ odr + patens + FE(code) + FE(year)

                1 year  2 years  4 years  5 years 
--------------------------------------------------
odr            0.0035   0.0035   0.0033   0.0033  
               (0.0041) (0.0043) (0.0047) (0.0050)
patens         0.0001   0.0001   0.0001   0.0001  
               (0.0001) (0.0001) (0.0001) (0.0001)
R-squared      0.7052   0.7115   0.7285   0.7368  
R-squared Adj. 0.6837   0.6729   0.6525   0.6399  
N              1120     560      280      224     
R2             0.705    0.712    0.728    0.737   
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


In [22]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model1 = smf.ols(
    "log_tfp ~ odr + log_internet_use + schooling + patens + C(code) + C(year)",
    data=df_est
)

results1 = model1.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results1 = pd.DataFrame({
    "coef": results1.params[["odr", "log_internet_use", "schooling", "patens"]],
    "std_err": results1.bse[["odr", "log_internet_use", "schooling", "patens"]],
    "p_value": results1.pvalues[["odr", "log_internet_use", "schooling", "patens"]],
})

print(main_results1.round(4))

# kopi af det færdige balanced panel
df_est = panel_2yr.copy()

# estimation
model2 = smf.ols(
    "log_tfp ~ odr + log_internet_use + schooling + patens + C(code) + C(year)",
    data=df_est
)

results2 = model2.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results2  = pd.DataFrame({
    "coef": results2.params[["odr", "log_internet_use", "schooling", "patens"]],
    "std_err": results2.bse[["odr", "log_internet_use", "schooling", "patens"]],
    "p_value": results2.pvalues[["odr", "log_internet_use", "schooling", "patens"]],
})

print(main_results2.round(4))

# kopi af det færdige balanced panel
df_est = panel_4yr.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + log_internet_use + schooling + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "log_internet_use", "schooling", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "schooling", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "schooling", "patens"]],
})

print(main_results3.round(4))

# kopi af det færdige balanced panel
df_est = panel_5yr.copy()

# estimation
model4 = smf.ols(
    "log_tfp ~ odr + log_internet_use + schooling + patens + C(code) + C(year)",
    data=df_est
)

results4 = model4.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results4 = pd.DataFrame({
    "coef": results4.params[["odr", "log_internet_use", "schooling", "patens"]],
    "std_err": results4.bse[["odr", "log_internet_use", "schooling", "patens"]],
    "p_value": results4.pvalues[["odr", "log_internet_use", "schooling", "patens"]],
})

print(main_results4.round(4))

                    coef  std_err  p_value
odr               0.0103   0.0045   0.0235
log_internet_use  0.0686   0.0223   0.0021
schooling        -0.0219   0.0137   0.1091
patens            0.0001   0.0001   0.0221
                    coef  std_err  p_value
odr               0.0103   0.0048   0.0307
log_internet_use  0.0696   0.0239   0.0036
schooling        -0.0226   0.0148   0.1266
patens            0.0001   0.0001   0.0257
                    coef  std_err  p_value
odr               0.0101   0.0053   0.0561
log_internet_use  0.0694   0.0259   0.0074
schooling        -0.0247   0.0169   0.1457
patens            0.0001   0.0001   0.0356
                    coef  std_err  p_value
odr               0.0101   0.0057   0.0737
log_internet_use  0.0701   0.0273   0.0103
schooling        -0.0254   0.0178   0.1541
patens            0.0002   0.0001   0.0332


In [25]:
results = summary_col(
    [results1, results2, results3, results4],
    stars=True,
    model_names=['1 year', '2 years', '4 years', '5 years'],
    regressor_order=['odr', 'log_internet_use', 'schooling', 'patens'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

#print the model
print("log(tfp) ~ odr + log_internet_use + schooling + patens + FE(code) + FE(year)")
print(results)

log(tfp) ~ odr + log_internet_use + schooling + patens + FE(code) + FE(year)

                   1 year   2 years   4 years  5 years 
-------------------------------------------------------
odr              0.0103**  0.0103**  0.0101*   0.0101* 
                 (0.0045)  (0.0048)  (0.0053)  (0.0057)
log_internet_use 0.0686*** 0.0696*** 0.0694*** 0.0701**
                 (0.0223)  (0.0239)  (0.0259)  (0.0273)
schooling        -0.0219   -0.0226   -0.0247   -0.0254 
                 (0.0137)  (0.0148)  (0.0169)  (0.0178)
patens           0.0001**  0.0001**  0.0001**  0.0002**
                 (0.0001)  (0.0001)  (0.0001)  (0.0001)
R-squared        0.7429    0.7485    0.7625    0.7699  
R-squared Adj.   0.7236    0.7137    0.6933    0.6813  
N                1120      560       280       224     
R2               0.743     0.749     0.763     0.770   
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01
